---

<div align="center">
  <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/python/python-original.svg" width="80"/>
</div>

<h1 align="center">GenAI & Advanced Nets: O Mecanismo de Atenção </h1>

<h3 align="center">PhD. Julles Mitoura</h3>

<div align="center">
  <img src="https://img.shields.io/badge/Python-3776AB?style=for-the-badge&logo=python&logoColor=white"/>
  <img src="https://img.shields.io/badge/Jupyter-F37626?style=for-the-badge&logo=jupyter&logoColor=white"/>
</div>

---

In [1]:
# Obs: se você não estiver utilizando um ambiente virtual, instale as bibliotecas conforme se apresenta abaixo
%pip install -q -r requirements.txt

# pip é o gerenciador de pacotes do Python. Pense nele como o instalador oficial de libs Python.
# no notebook, usar %pip ... é ideal porque instala no mesmo ambiente do kernel em uso.

# -q: quiet
# -r: requirement file, indica ao pip para instalar os pacotes listados no arquivo requirements.txt


[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---

<div align="center">

## <span style="color:#1E90FF;">Anatomia da Arquitetura Transformer do Zero</span>

</div>

Na etapa anterior, desenvolvemos a arquitetura Transformer com PyTorch, aproveitando recursos prontos da biblioteca. Agora, o objetivo é implementar em Python, do zero, os principais módulos dessa arquitetura.

No modelo anterior, utilizamos o `nn.MultiheadAttention`. Embora pareça uma chamada simples, esse módulo do `torch.nn` encapsula várias operações internas. A Figura 1 ajuda a visualizar essa diferença entre a implementação pronta e a construção manual do mecanismo de atenção.

<div style="text-align: center;">
<p><em>Figura 1: Single-Head-Attention e Multi-Head Attention.</em></p>
  <img src="imgs/img03.png" alt="Arquitetura de Transformadores" width="500"/>
</div>

O objetivo agora será desenvolver em Python uma `Single-Head Attention`. Com isso, teremos mais visão sobre os detalhes desta arquitetura e poderemos seguir com o desenvolvimento.

---

Objetivo: Criar um modelo capaz de prever sequencias de comprimento de 1 a 10 tokens.

O modelo transformer será constituido por:
1. Camada de embedding
2. Mecanismo de atenção
3. Camandas Encoder e Decoder
4. Camadas Linear e Softmax

### 1. Hiperparâmetros Iniciais

In [2]:
# imports fundamentis
import numpy as np

# dimensão do modelo
dim_model = 64
# comprimento da sequencia que será predita pelo modelo
seq_length = 10
# tamanho do vocabulario
vocab_size = 100   

#### `dim_model` (dimensão do modelo)

Define o tamanho dos vetores internos usados para representar cada token (embeddings e estados ocultos).

- **Se aumentar `dim_model`**:
  - melhora a capacidade de representação;
  - aumenta custo computacional e uso de memória;
  - pode elevar risco de overfitting em bases pequenas.
- **Se diminuir `dim_model`**:
  - reduz custo e acelera treino/inferência;
  - pode limitar a capacidade do modelo (underfitting).

#### `seq_length` (comprimento da sequência)

Define quantos tokens o modelo observa por amostra (janela de contexto).

- **Se aumentar `seq_length`**:
  - permite capturar dependências mais longas no contexto;
  - aumenta bastante o custo da atenção (aprox. quadrático em relação ao tamanho da sequência);
  - exige mais memória.
- **Se diminuir `seq_length`**:
  - torna o treino mais rápido e barato;
  - reduz o contexto disponível para predição.

#### `vocab_size` (tamanho do vocabulário)

Define quantos tokens distintos existem no espaço de entrada/saída do modelo.

- **Se aumentar `vocab_size`**:
  - melhora cobertura de tokens possíveis;
  - aumenta o tamanho da camada de saída e o custo do treinamento.
- **Se diminuir `vocab_size`**:
  - reduz custo computacional;
  - pode perder informação e aumentar ambiguidades (mais colisões de token).

### 2. Camada de Embeddings

A função de embedding tem por objetivo converter entradas sequenciais em ventores densos de tamanho fixo. Esses vetores são conhecidos como embeddings e são parte fundamental da arquiteture que estamos desenvolvendo.

In [3]:
# função para criar embeddings de uma sequência de tokens
# entrada: lista/array de IDs de token
# saída: matriz (seq_length, dim_model)

def embeddings(input_ids, vocab_size, dim_model):
    # crie uma matriz de embedding onde cada linha representa um token do vocabulário
    # a matriz é iniciada com valores aleatórios normalmente distribuidos
    embed = np.random.randn(vocab_size, dim_model)

    # para cada indice do token input, seleciona-se o embedding correspondente da matriz
    # retorna um array de embeddings correspondentes à sequencia de entrada
    return np.array([embed[i] for i in input_ids])


# exemplo de uso
input_exemplo = [3, 10, 7, 25, 1]
saida_embeddings = embeddings(input_exemplo, vocab_size=vocab_size, dim_model=dim_model)

print("Tokens de entrada:", input_exemplo)
print("Shape da saída:", saida_embeddings.shape)  # (5, 64)
print("Primeiro vetor de embedding (token 3):")
print(saida_embeddings[0])

Tokens de entrada: [3, 10, 7, 25, 1]
Shape da saída: (5, 64)
Primeiro vetor de embedding (token 3):
[-0.68554861 -0.45152296  0.14414857 -0.65058801 -0.63668739  0.30716436
  0.58857942 -0.21801664  0.11659075 -1.03204254  0.35670033 -0.09734825
 -1.33811574  0.05852206  0.05720779 -0.05639332 -1.98158427 -0.9089521
  0.81693208 -0.58764406 -0.13513163 -1.10299567 -0.16814482 -0.18236821
 -1.03152824  1.45369992  0.75398049  0.54122242 -0.48401851  1.24006925
 -0.57797502 -0.38976424  0.17252394 -1.61274839  0.99859696  0.25210731
 -2.27846056 -1.28499678 -0.55601376  1.09031585 -1.11485435  0.40621774
  0.50886775 -0.22884433  0.60174427  2.25396021  0.24054458 -1.40732785
 -1.81113004 -0.42325421 -0.89973843  0.51330269  0.89581588 -0.48108443
  0.44870593  0.71471107 -1.96011522  0.36854945 -0.5946321  -1.34607721
 -0.09156093 -0.57816238  0.77666228 -0.40352835]


### 3. Camada de Atenção

Buscaremos construir o mecanismo de atenção como é apresentado na Figura 2.

<div style="text-align: center;">
<p><em>Figura 2: Single-Head-Attention.</em></p>
  <img src="imgs/img04.png" alt="Arquitetura de Transformadores" width="300"/>
</div>

No Transformer, os vetores `Q`, `K` e `V` são projeções lineares usadas para calcular atenção, mas a origem deles depende da subcamada:

- **Self-attention no encoder:** `Q`, `K` e `V` vêm da mesma entrada (`X`);
- **Self-attention no decoder:** `Q`, `K` e `V` também vêm da mesma entrada do decoder (com máscara causal);
- **Cross-attention no decoder:** `Q` vem da saída da subcamada anterior do decoder, enquanto `K` e `V` vêm da saída do encoder.

Antes das projeções, $X$ representa a matriz de entrada da atenção (embeddings dos tokens, geralmente já combinados com informação posicional). Se a sequência possui $n$ tokens e dimensão $d_{\text{model}}$, então:

$$
X \in \mathbb{R}^{n \times d_{\text{model}}}
$$

Formalmente:

$$
\begin{aligned}
Q &= XW_Q \\
K &= XW_K \\
V &= XW_V
\end{aligned}
$$

A atenção é calculada por:

$$
\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

em que os escores entre `Q` e `K` definem os pesos aplicados sobre `V`.

### 4. Construção do Método Softmax

A função softmax é uma função de ativação amplamente utilizada em redes neurais, especialmente em cenários de classificação, nos quais é importante transformar valores brutos de saída (*logits*) em probabilidades que somam 1.

No contexto de atenção, ela converte os escores em pesos normalizados, destacando os elementos mais relevantes.

A formulação matemática da softmax para cada elemento $i$ é:

$$
\mathrm{softmax}(x_i) = \frac{e^{x_i}}{\sum_{j=1}^{n} e^{x_j}}
$$

Abaixo, está a implementação da função softmax com comentários em cada etapa do cálculo.

In [5]:
# função softmax

def softmax(x):

    # calcula a exponencial de cada elemento do input ajustado pelo maximo valor do input
    # para evitar overflow numerico
    e = np.exp(x-np.max(x))

    # divide cada exponencial pelo somatorio dos exponenciais ao longo do ultimo aixo (axis = 1)
    # o reshape(-1,1) garante que a divisao seja realizada corretamente em um contexto multidimensional
    return e/e.sum(axis=-1).reshape(-1,1)

### 5. Scale Dot Product

A função `scaled_dot_product_attention()` é um componente central do mecanismo de atenção em modelos Transformer. Ela calcula a atenção entre conjuntos de *queries* (`Q`), *keys* (`K`) e *values* (`V`).

Em termos práticos, essa função calcula escores com produto escalar entre `Q` e `K`, aplica a escala por $\sqrt{d_k}$, utiliza `softmax` para obter pesos normalizados e combina esses pesos com `V`.

Esse processo permite que o modelo atribua diferentes níveis de importância às partes da entrada, tornando os Transformers especialmente eficazes em tarefas de PLN e outras tarefas sequenciais.

In [7]:
# escrevendo a função scale dot product
def scale_dot_product_attention(Q, K, V):

    # calcula o produto escalar entre Q e a transposta de K
    matmul_qk = np.dot(Q, K.T)

    # obtem a dimensão dos vetores de chave
    depth = K.shape[-1]

    # escala os logits dividindo-os pela raiz quadrada da profundidade
    logits = matmul_qk/np.sqrt(depth)

    # aplica a função softmax para obter os pesos de atenção
    attention_values = softmax(logits)

    # multiplica os pesos de atenção pelos valores V para obter a saida final

    out = np.dot(attention_values, V)
    
    return out